In [ ]:
import os, sys, torch
import torch.nn.functional as F
import pandas as pd, re, random
import numpy as np
sys.path.insert(0, r"c:\repos\DroneDetectionRF")


from NoisyUAV.funciones.cargador import cargar_muestra, NOMBRES_CLASES
from NoisyUAV.funciones.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico
from NoisyUAV.modelos.burst_cvcnn import BurstCVCNN

In [ ]:
device = torch.device("cpu")
# IMPORTANTE: Cargamos el modelo en el que PyTorch está trabajando AHORA MISMO
ruta_pesos = r"c:\repos\DroneDetectionRF\NoisyUAV\resultados_burst_v2\checkpoints\best_model.pt"
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model = BurstCVCNN().to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
# Rescatamos las estadísticas físicas de la base de datos original
phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32).to(device)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32).to(device)
print(f"✅ Nuevo modelo BurstCVCNN cargado con éxito en {device.type.upper()}")
print(f"   Época Óptima del guardado: {ckpt.get('epoch', 'N/A')}")

In [ ]:
# ===== PARAMETROS AJUSTABLES =====
TARGET_DESEADO = 6      # 0=DJI, 1=FutabaT14, 2=FutabaT7, 3=Graupner, 5=Taranis, 6=Turnigy
SNR_DESEADA    = -4     
# ==================================

In [ ]:
# 1. EMPAREJAMIENTO ESTRICTO CON LA DIVISIÓN DE TEST (Burst Dataset)
CSV_NUEVO = r"c:\repos\DroneDetectionRF\NoisyUAV\resultados_burst_v2\bursts_dataset.csv"
df = pd.read_csv(CSV_NUEVO)

In [ ]:
# Cogemos ÚNICAMENTE las señales congeladas como test
df_test = df[df['split'] == 'test'].copy()
df_padres_test = df_test.drop_duplicates(subset=['file_path']).copy()
candidatos = df_padres_test[df_padres_test["label"] == 1].copy()   # solo drones

In [ ]:
if TARGET_DESEADO is not None:
    candidatos = candidatos[candidatos["target_multiclass"] == TARGET_DESEADO]
    if candidatos.empty:
        raise ValueError(f"❌ No hay muestras de target={TARGET_DESEADO} en el test set.")
if SNR_DESEADA is not None:
    candidatos = candidatos[candidatos["snr"] == SNR_DESEADA]

In [ ]:
nombre_clase = NOMBRES_CLASES.get(TARGET_DESEADO, f"target{TARGET_DESEADO}") if TARGET_DESEADO is not None else "cualquier dron"
print(f"✅ {len(candidatos)} señales disponibles vírgenes → [{nombre_clase} | SNR={SNR_DESEADA} dB]")

In [ ]:
semilla_aleatoria = 495
# semilla_aleatoria = random.randint(0, 9999)

sample_seleccionado = candidatos.sample(1, random_state=semilla_aleatoria).iloc[0]
parent_filename = sample_seleccionado["file_path"]
RUTA_RAW_PARENT = os.path.join(r"C:\TFM_data\NoisyUAV\drone_RF_data", parent_filename)
print("=" * 52)
print(f"🎬 Muestra Ciega Recuperada: {parent_filename} (Semilla {semilla_aleatoria})")
print("=" * 52)

In [ ]:
RUTA_RAW_PARENT

In [ ]:
iq_tensor, _, original_target, original_snr = cargar_muestra(RUTA_RAW_PARENT)
print(f"✅ Tensor cargado: {iq_tensor.shape}")

In [ ]:
# Variables Físicas de la Tesis
FS = 14e6
NPERSEG = 2048
Z_THRESH = 3.5      
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 2.5
MIN_Z_ABS = 4.0
BG_MULT = 4
MAX_BINS_FRAC = 0.25
SMOOTH_MS = 0.2
ADAPTIVE_WINDOW_MS = 10 
# Detección
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target=f"Target {original_target}", snr=f"{original_snr}", index=0,
)
# Magia Visual de tu Proyecto
fig_2d = plot_muestra(
    iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
    titulo=f"Validación End-To-End: {parent_filename}"
)
fig_2d.show()

In [ ]:
print("==============================================")
print("  VEREDICTO BURST-CVCNN (NUEVA ARQUITECTURA) ")
print("==============================================")
drones_encontrados = 0
ruidos_encontrados = 0
if len(bursts) == 0:
    print("  ❌ No se detectaron ráfagas. La IA asume que la sala está vacía (RUIDO).")
else:
    # Parámetros Globales estáticos para toda la sala
    global_nf     = float(np.median(nf_v))
    global_ns     = float(np.clip(ns, 0, 5))
    global_H_mean = float(np.mean(H_smooth))
    global_p75_act= float(np.percentile(n_active, 75))
    with torch.no_grad():
        for i, b in enumerate(bursts):
            # 1. RECORTAR LA ONDA EXACTA
            idx_inicio = int(b['t0'] * 1e-3 * FS)
            idx_fin = int(b['t1'] * 1e-3 * FS)
            if idx_inicio >= idx_fin:
                continue
            pulso = iq_tensor[:, idx_inicio:idx_fin]
            
            # Normalización RMS de este latido
            power = pulso.pow(2).mean().clamp(min=1e-12).sqrt()
            pulso_normalizado = pulso / power
            input_ia = pulso_normalizado.unsqueeze(0).to(device) 
            
            # 2. CONSTRUIR EL PERFIL FÍSICO (8 Dimensiones)
            dur_ms      = np.clip(b['dur_ms'], 0, 75)
            z_peak      = np.clip(abs(b['z_peak']), 0, 30)
            drop_b      = np.clip(b['drop_b'], 0, 10)
            n_act_burst = np.clip(b['n_act'], 0, 2048)
            
            feat_array = np.array([dur_ms, z_peak, drop_b, n_act_burst, 
                                   global_nf, global_ns, global_H_mean, global_p75_act], dtype=np.float32)
            
            feat_t = torch.from_numpy(feat_array).to(device)
            feat_norm = torch.clamp((feat_t - phys_mean) / (phys_std + 1e-8), -5.0, 5.0).unsqueeze(0)
            
            # 3. JUZGADO HÍBRIDO (El Cerebro)
            logit = model(input_ia, feat_norm)
            prob_dron = torch.sigmoid(logit).item() * 100 
            
            t_ms_inicio = b['t0']
            if prob_dron > 50.0:
                drones_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
            else:
                ruidos_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")
    print("----------------------------------------------")
    print(f"Resumen de la sala: {drones_encontrados} Drones | {ruidos_encontrados} Falsas alarmas")